# Embeddings + FAISS

In [ ]:
pip install faiss-cpu sentence-transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
from pathlib import Path

DATA_PATH = Path("/content/drive/MyDrive/Data/processed/chunks.jsonl")

chunks = []

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"Loaded {len(chunks)} chunks")

Loaded 17590 chunks


## EMBEDDING MODEL

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-base-en-v1.5")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## PREPARE TEXT FOR EMBEDDING

In [ ]:
def build_embedding_text(chunk):
    return f"""
Title: {chunk['section']}
Content: {chunk['text']}
"""

## GENERATE EMBEDDINGS

In [ ]:
from tqdm import tqdm
import numpy as np

texts = [build_embedding_text(c) for c in chunks]

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # IMPORTANT for cosine similarity
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/550 [00:00<?, ?it/s]

Embeddings shape: (17590, 768)


## BUILD FAISS INDEX

In [ ]:
import faiss

dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)  # cosine similarity (because normalized)
index.add(embeddings)

print("FAISS index size:", index.ntotal)

FAISS index size: 17590


## SAVE INDEX + METADATA

In [ ]:
SAVE_DIR = Path("/content/drive/MyDrive/Data/vector_store")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# save FAISS
faiss.write_index(index, str(SAVE_DIR / "faiss.index"))

# save metadata
with open(SAVE_DIR / "chunks_meta.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False)

print("✅ Saved vector DB")

✅ Saved vector DB


## REMOVE DUPLICATES

In [ ]:
def deduplicate_results(results, similarity_threshold=0.85):
    unique = []

    for r in results:
        is_duplicate = False
        for u in unique:
            if r["text"][:200] == u["text"][:200]:
                is_duplicate = True
                break

        if not is_duplicate:
            unique.append(r)

    return unique

## RETRIEVAL FUNCTION `RETURN PARENT CONTEXT`

In [ ]:
def retrieve(query, top_k=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, top_k * 2)

    results = []
    for i, idx in enumerate(indices[0]):
        chunk = chunks[idx]

        results.append({
            "score": float(scores[0][i]),
            "text": chunk["text"],
            "section": chunk["section"],
            "doc_id": chunk["doc_id"],
            "parent_text": chunk["parent_text"]
        })

    # ✅ remove duplicates
    results = deduplicate_results(results)

    # ✅ keep top_k only
    return results[:top_k]

## REMOVE DUPLICATE HEADERS

In [ ]:
def clean_context_text(text):
    lines = text.split("\n")
    seen = set()
    cleaned = []

    for line in lines:
        line_strip = line.strip()

        # remove duplicate headers
        if line_strip.startswith("###"):
            if line_strip in seen:
                continue
            seen.add(line_strip)

        cleaned.append(line)

    return "\n".join(cleaned)

## MERGE SIMILAR SECTIONS

In [ ]:
def merge_sections(results):
    merged = {}

    for r in results:
        key = r["section"].lower()

        if key not in merged:
            merged[key] = r["parent_text"]
        else:
            merged[key] += "\n" + r["parent_text"]

    return list(merged.values())

## REMOVE HEADERS

In [ ]:
def strip_headers(text):
    return "\n".join([
        line for line in text.split("\n")
        if not line.strip().startswith("#")
    ])

## BUILD CLEAN CONTEXT

In [ ]:
def build_context(results):

    merged_sections = merge_sections(results)

    cleaned_blocks = []

    for sec in merged_sections:
        sec = clean_context_text(sec)
        sec = strip_headers(sec)

        # limit size per section
        cleaned_blocks.append(sec[:1200])

    # 🔥 merge everything into ONE coherent block
    context = "\n\n".join(cleaned_blocks)

    return context.strip()

## TEST RETRIEVAL

In [ ]:
query = "Explain transformer architecture clearly"

results = retrieve(query, top_k=5)

context = build_context(results)

print(context[:2000])

The transformer model was primarily developed based on the attention mechanism (Vaswani et al., 2017), with the aim of processing sequential data. Its outstanding performance, especially in achieving state-of-the-art benchmarks for NLP translation models, has led to the widespread use of transformers. As depicted in

The Transformer (Vaswani et al., 2017) employs an encoder-decoder structure, consisting of stacked encoder and decoder layers. Encoder layers consist of two sublayers: self-attention followed by a position-wise feed-forward layer. Decoder layers consist of three sublayers: selfattention followed by encoder-decoder attention, followed by a position-wise feed-forward layer. It uses residual connections around each of the sublayers, followed by layer normalization (Ba et al., 2016). The decoder uses masking in its selfattention to prevent a given output position from incorporating information about future output positions during training.
Position encodings based on sinusoids

In [ ]:
def build_prompt(query, context):

    prompt = f"""
You are an expert in Generative AI.

Answer the following question clearly and in a structured way.

Use the provided context, but rewrite it into a clean explanation.

Question:
{query}

Context:
{context}

Instructions:
- Do NOT repeat text
- Do NOT mention sections
- Explain step by step
- Use simple clear language
- Include key components and how they work

Answer:
"""
    return prompt

# Reload Vector DB + Model + Functions

In [ ]:
pip install faiss-cpu sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================
# LOAD EVERYTHING (NO REBUILD)
# =========================

import json
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

# paths
SAVE_DIR = Path("/content/drive/MyDrive/Data/vector_store")

FAISS_PATH = SAVE_DIR / "faiss.index"
META_PATH  = SAVE_DIR / "chunks_meta.json"

# load FAISS
index = faiss.read_index(str(FAISS_PATH))

# load metadata (chunks)
with open(META_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"✅ Loaded FAISS index with {index.ntotal} vectors")
print(f"✅ Loaded {len(chunks)} chunks")

# load embedding model (IMPORTANT: same model used before)
model = SentenceTransformer("BAAI/bge-base-en-v1.5")

print("✅ Model loaded")

✅ Loaded FAISS index with 17590 vectors
✅ Loaded 17590 chunks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded


In [ ]:
# =========================
# DEDUPLICATION
# =========================
def deduplicate_results(results, similarity_threshold=0.9):
    unique = []

    for r in results:
        is_duplicate = False

        for u in unique:
            # stronger check (semantic + text)
            if r["section"].lower() == u["section"].lower():
                is_duplicate = True
                break

            if r["text"][:150] == u["text"][:150]:
                is_duplicate = True
                break

        if not is_duplicate:
            unique.append(r)

    return unique


# =========================
# RETRIEVE
# =========================
def retrieve(query, top_k=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, top_k * 5)  # 🔥 increase pool

    results = []
    seen_sections = set()

    for i, idx in enumerate(indices[0]):
        if idx == -1:
            continue

        chunk = chunks[idx]
        section = chunk["section"].lower()

        # 🔥 enforce diversity
        if section in seen_sections:
            continue

        seen_sections.add(section)

        results.append({
            "score": float(scores[0][i]),
            "text": chunk["text"],
            "section": chunk["section"],
            "doc_id": chunk["doc_id"],
            "parent_text": chunk["parent_text"]
        })

        if len(results) >= top_k:
            break

    return results


# =========================
# CLEAN CONTEXT
# =========================
def clean_context_text(text):
    lines = text.split("\n")
    seen = set()
    cleaned = []

    for line in lines:
        line_strip = line.strip()

        if line_strip.startswith("###"):
            if line_strip in seen:
                continue
            seen.add(line_strip)

        cleaned.append(line)

    return "\n".join(cleaned)


def merge_sections(results):
    merged = {}

    for r in results:
        key = r["section"].lower()

        if key not in merged:
            merged[key] = r["parent_text"]
        else:
            merged[key] += "\n" + r["parent_text"]

    return list(merged.values())


def strip_headers(text):
    return "\n".join([
        line for line in text.split("\n")
        if not line.strip().startswith("#")
    ])


def build_context(results):

    sections = []

    for r in results:
        sec = r["parent_text"]

        sec = clean_context_text(sec)
        sec = strip_headers(sec)

        # add section title explicitly
        formatted = f"{r['section'].upper()}:\n{sec[:1000]}"
        sections.append(formatted)

    # join nicely
    context = "\n\n".join(sections)

    return context.strip()

In [ ]:
query = "Explain transformer architecture clearly"

results = retrieve(query, top_k=5)

print("🔍 Retrieved Chunks:\n")
for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} (score={r['score']:.4f}) ---")
    print(r["section"])
    print(r["text"][:300])


print("\n\n🧠 FINAL CONTEXT:\n")

context = build_context(results)

print(context[:2000])

🔍 Retrieved Chunks:


--- Result 1 (score=0.7786) ---
2.2 Architecture Of The Transformer Model
### 2.2 Architecture Of The Transformer Model The transformer model was primarily developed based on the attention mechanism (Vaswani et al., 2017), with the aim of processing sequential data. Its outstanding performance, especially in achieving state-of-the-art benchmarks for NLP translation models

--- Result 2 (score=0.7616) ---
2.1 Transformer
### 2.1 Transformer The Transformer (Vaswani et al., 2017) employs an encoder-decoder structure, consisting of stacked encoder and decoder layers. Encoder layers consist of two sublayers: self-attention followed by a position-wise feed-forward layer. Decoder layers consist of three sublayers: selfat

--- Result 3 (score=0.7548) ---
3.2. Architectures
### 3.2. Architectures While the Transformer was originally introduced with an encoder-decoder architecture, much modern work on transfer learning for NLP uses alternative architectures. In this sectio

In [ ]:
print("FAISS dimension:", index.d)

FAISS dimension: 768


In [ ]:
print("Model dimension:", model.get_sentence_embedding_dimension())

Model dimension: 768


/tmp/ipykernel_1803/68408981.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model dimension:", model.get_sentence_embedding_dimension())
